# Aurora Forecasts - Demand Processing

## Overview

This notebook processes yearly electricity demand forecasts including total demand and demand breakdowns by sector.

### Key Functions:
- Extracts total electricity demand forecasts
- Processes EV (electric vehicle) demand
- Processes electric heat demand
- Processes emerging demand (electrolyzers, data centers)
- Converts units (TWh to GWh)

### Demand Categories:
Total Demand, EV Demand, Electric Heat, Emerging Demand


## 1. Import Dependencies

Import required libraries for demand data processing.

In [9]:
import pandas as pd
from typing import Tuple
from datetime import datetime

In [ ]:
# Paths

tracker_path = '../../../outputs/trackers/demand_tracker.xlsx'
path_technology = '../../../data/aurora/aurora2_technology_scenarios_ES_default_currency_1y.parquet'
path_system = '../../../data/aurora/aurora2_system_scenarios_ES_default_currency_1y.parquet'
sharepoint_folder = "Market Research/Trackers - EUROPE/Generation Mix/Backup/Python"

In [11]:
country = 'Spain'
year = datetime.now().year
commodity_list = ['Baseload price', 'Carbon price', 'Gas price', 'Coal price']
# Take into account that we are manually filtering the df_system2 to only include commodity price variables in the box above.
# As Aurora updates its API, we should check if they have included more commodity price variables

In [12]:
# We define the release we want to extract data from

release = '26Q2' # '25Q4', '26Q1', '26Q2'
# Mapping for country codes in Aurora to country names

from common_libs.utils.utils_dates import get_quarterly_strings

quarter_dict, quarter_year, quarter_month = get_quarterly_strings(quarterly=release)

In [13]:
demand_list = ['Total demand', 'EV demand', 'Electric heat demand', 'Electrolyser demand', 'Data centre demand']
peak_demand_list = ['Peak inflexible demand']

In [14]:
df_system2 = pd.read_parquet(path_system)

In [15]:
df = df_system2[((df_system2['variable'].str.lower().isin([x.lower() for x in demand_list])) | (df_system2['variable'].str.lower().str.contains('demand'))) & (~df_system2['variable'].str.lower().isin([x.lower() for x in peak_demand_list]))]

df['variable'].unique()

array(['Total demand', 'EV demand', 'Peak demand', 'Electric heat demand',
       'Electrolyser demand', 'Data centre demand'], dtype=object)

In [21]:
# Adapt format to tracker format

df_fil = df[
    (df['sensitivity'].isin(['central', 'low', 'high'])) &
    (df['name'] != 'Great Britain Jul 22 Locational Balancing (Central)') # we are excluding this specific release since it is the only one with duplicated rows based on the pivot index + year + value, which is likely an error in the data that we should clarify with Aurora before including it in the tracker
    ].reset_index(drop=True)
        

df_fil.sort_values(by=['name', 'year', 'variable'], inplace=True)

df_fil['Source'] = 'Aurora'
df_fil['Scenario'] = df_fil['sensitivity'].str.capitalize()

from aurora_forecasts.utils.forecast_release import parse_release_info  # bring in helper to parse release metadata from the release name

# Derive structured release info for each row so we can pull out quarter, month, and year later in the pipeline
df_fil['release'] = df_fil['name'].apply(parse_release_info)


# Transform demand data from TWh to GWh

df_fil['value'] = df_fil['value'].astype(float) * 1_000

from aurora_forecasts.processing.dicts import country_tracker_map, region_tracker_map, demand_drivers_tracker_mappings

df_fil['country'] = df_fil['region_code'].map(country_tracker_map)
df_fil['region'] = df_fil['region_code'].map(region_tracker_map)
df_fil['variable'] = df_fil['variable'].map(demand_drivers_tracker_mappings)

if df_fil['variable'].isnull().any():
    raise ValueError(f"Null values found in 'variable' column after mapping. Please check the following unmapped variables: {df_fil[df_fil['variable'].isnull()]['variable'].unique()}")


df_fil['variable'] = df_fil['variable'].str.replace('Solar', 'Solar PV', regex=False)
df_fil['currency'] = df_fil['currency'].str.slice(0, -4).str.upper()

# Extract the quarter number to support filtering/grouping by release period
df_fil["release_quarter"] = df_fil["release"].apply(lambda x: x.quarter)

# Pull the release month, which may be needed for tracker formatting or consistency checks
df_fil["release_month"]   = df_fil["release"].apply(lambda x: x.month)

# Pull the string month, which will be needed afterwards for tracker formatting
df_fil['string_month']    = df_fil["release"].apply(lambda x: x.month_str)

# Capture the release year so that every row carries the release publication year alongside the actual data year
df_fil["release_year"]    = df_fil["release"].apply(lambda x: x.year)

# Build a composite release identifier (e.g. '2025-Oct') used for grouping rows in the tracker
df_fil['release_string'] = df_fil['release_year'].astype(str) + '/' + df_fil['string_month']

# Capture the release year so that every row carries the release publication year alongside the actual data year
df_fil["release_year"]    = df_fil["release"].apply(lambda x: x.year)
df_fil['date'] = df_fil['release_year'].astype(str) + '/' + df_fil['release_month'].astype(str)

columns = df_fil.columns.tolist()
columns.remove('year')
columns.remove('value')

pivot_index = ['region', 'Source', 'release_string', 'Scenario', 'variable',  
    'units']

df_fil_pivotted = df_fil.pivot(
    index=pivot_index,
    columns='year',
    values='value'
).reset_index()

# We add empty columns from data since 2016, in order to match the tracker format
df_fil_pivotted_columns = df_fil_pivotted.columns.tolist()
for year in range(2016, df_fil['year'].max()+1):
    if year not in df_fil_pivotted_columns:
        df_fil_pivotted[year] = pd.NA

# we now resort the dataframe
df_fil_pivotted = df_fil_pivotted.reindex(
    columns= pivot_index + list(range(2016, df_fil['year'].max()+1))
)

df_fil_pivotted.sort_values(by=pivot_index, inplace=True)

df_fil_pivotted.reset_index(drop=True, inplace=True)

df_fil_pivotted

year,region,Source,release_string,Scenario,variable,units,2016,2017,2018,2019,...,2051,2052,2053,2054,2055,2056,2057,2058,2059,2060
0,France,Aurora,2020/Oct,Central,Electric Vehicles,TWh,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,France,Aurora,2020/Oct,Central,Electricity demand,TWh,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,France,Aurora,2020/Oct,Central,Peak demand,GW,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,France,Aurora,2020/Oct,High,Electricity demand,TWh,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,France,Aurora,2020/Oct,Low,Electricity demand,TWh,<NA>,<NA>,<NA>,<NA>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3390,Spain,Aurora,2026/Jan,Low,Data centre,TWh,<NA>,<NA>,<NA>,<NA>,...,25800.0,25800.0,25900.0,26000.0,26100.0,26200.0,26200.0,26300.0,26400.0,26500.0
3391,Spain,Aurora,2026/Jan,Low,Electric Vehicles,TWh,<NA>,<NA>,<NA>,<NA>,...,40800.0,41600.0,42400.0,43200.0,43800.0,44400.0,44900.0,45400.0,45900.0,46200.0
3392,Spain,Aurora,2026/Jan,Low,Electric heating,TWh,<NA>,<NA>,<NA>,<NA>,...,20100.0,20700.0,21200.0,21700.0,22300.0,22800.0,23300.0,23800.0,24300.0,24900.0
3393,Spain,Aurora,2026/Jan,Low,Electricity demand,TWh,<NA>,<NA>,<NA>,<NA>,...,382300.0,387400.0,391800.0,396300.0,402300.0,407500.0,411300.0,415200.0,418800.0,422500.0


In [17]:
df_fil_pivotted['variable'].unique()

array(['Electric Vehicles', 'Electricity demand', 'Peak demand',
       'Electric heating', 'Electrolysis', 'Data centre'], dtype=object)

In [18]:
df__ = df_fil_pivotted[df_fil_pivotted['release_string'] == '2026/Apr']
df__

year,region,Source,release_string,Scenario,variable,units,2016,2017,2018,2019,...,2051,2052,2053,2054,2055,2056,2057,2058,2059,2060
216,France,Aurora,2026/Apr,Central,Data centre,TWh,<NA>,<NA>,<NA>,<NA>,...,37600.0,37800.0,37900.0,38000.0,38200.0,38400.0,38400.0,38600.0,38700.0,38900.0
217,France,Aurora,2026/Apr,Central,Electric Vehicles,TWh,<NA>,<NA>,<NA>,<NA>,...,92400.0,93300.0,94000.0,94200.0,94400.0,94600.0,94800.0,94900.0,95000.0,95000.0
218,France,Aurora,2026/Apr,Central,Electric heating,TWh,<NA>,<NA>,<NA>,<NA>,...,50400.0,50300.0,50200.0,50200.0,50100.0,49800.0,49900.0,49900.0,49800.0,49700.0
219,France,Aurora,2026/Apr,Central,Electricity demand,TWh,<NA>,<NA>,<NA>,<NA>,...,618400.0,621500.0,623700.0,625900.0,627900.0,630100.0,631900.0,633900.0,635800.0,637800.0
220,France,Aurora,2026/Apr,Central,Electrolysis,TWh,<NA>,<NA>,<NA>,<NA>,...,50100.0,50800.0,51200.0,51700.0,52200.0,52900.0,53300.0,53900.0,54400.0,54900.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3375,Spain,Aurora,2026/Apr,Low,Data centre,TWh,<NA>,<NA>,<NA>,<NA>,...,25800.0,25800.0,25900.0,26000.0,26100.0,26200.0,26200.0,26300.0,26400.0,26500.0
3376,Spain,Aurora,2026/Apr,Low,Electric Vehicles,TWh,<NA>,<NA>,<NA>,<NA>,...,40800.0,41600.0,42400.0,43200.0,43800.0,44400.0,44900.0,45400.0,45900.0,46200.0
3377,Spain,Aurora,2026/Apr,Low,Electric heating,TWh,<NA>,<NA>,<NA>,<NA>,...,20100.0,20700.0,21200.0,21700.0,22300.0,22800.0,23300.0,23800.0,24300.0,24900.0
3378,Spain,Aurora,2026/Apr,Low,Electricity demand,TWh,<NA>,<NA>,<NA>,<NA>,...,367900.0,372800.0,377100.0,381400.0,385400.0,389800.0,393400.0,397100.0,400600.0,404900.0


In [19]:
df_fil_pivotted.to_excel(tracker_path, sheet_name='demand', index=False)

In [ ]:
# Authenticate with Microsoft Graph and upload the tracker Excel to SharePoint.
# Credentials are loaded from YAML config + environment variables — never hardcoded.
from pathlib import Path
from common_libs.sp.settings import load_ms_config
from common_libs.sp.graph_client import GraphSharePointClient
import os

sp_base_path = os.getenv("SP_BASE_PATH")  # Example env variable for base path; adjust name as needed

# Extract just the filename from the local tracker path for use as the SharePoint upload target
tracker_path = Path(tracker_path).name

sharepoint_target_path = f"{sp_base_path}/{sharepoint_folder}/{tracker_path}"

df_fil_pivotted.to_excel(sharepoint_target_path, index=False)